<a href="https://colab.research.google.com/github/LBT-975/BTL_TTNT_Nhom6/blob/main/TTNT_Nhom6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code chỉ chứa mô hình của mình

In [ ]:
# ============================================================================
# TIỀN XỬ LÝ DỮ LIỆU PHÂN TÍCH CẢM XÚC TIẾNG VIỆT
# ============================================================================

import pandas as pd
import re
import unicodedata
import time
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '/content/data.csv'
OUTPUT_PATH = 'cleaned_data.csv'

USE_STOPWORDS = False


# ============================================================================
# 1. TỪ ĐIỂN EMOJI → VĂN BẢN (Emoji Semantics Translation)
# ============================================================================
EMOJI_MAP = {
    # --- Tích cực (Positive) ---
    '❤️': ' tốt ', '❤': ' tốt ', '💖': ' tốt ', '💕': ' tốt ',
    '💗': ' tốt ', '💓': ' tốt ', '💘': ' tốt ', '💝': ' tốt ',
    '💞': ' tốt ', '🥰': ' tốt ', '😍': ' tốt ', '😘': ' tốt ',
    '😻': ' tốt ', '💋': ' tốt ', '🫶': ' tốt ',


    '👍': ' tốt ', '👍🏻': ' tốt ', '👍🏼': ' tốt ',
    '👍🏽': ' tốt ', '👍🏾': ' tốt ', '👍🏿': ' tốt ',
    '👏': ' tốt ', '👏🏻': ' tốt ', '🙌': ' tốt ',
    '✅': ' tốt ', '✔️': ' tốt ', '☑️': ' tốt ',
    '💯': ' tốt ', '🎉': ' tốt ', '🎊': ' tốt ',
    '🤝': ' tốt ', '💪': ' tốt ',


    '🤩': ' tốt ', '🔥': ' tốt ', '⚡': ' tốt ',
    '🏆': ' tốt ', '🥇': ' tốt ', '🎯': ' tốt ',


    '😊': ' vui ', '🙂': ' vui ', '😄': ' vui ', '😃': ' vui ',
    '😁': ' vui ', '😆': ' vui ', '🤗': ' vui ', '😜': ' vui ',
    '🤪': ' vui ', '😝': ' vui ', '😋': ' vui ', '😎': ' vui ',
    '😏': ' vui ', '🙈': ' vui ', '🤭': ' vui ', '😌': ' vui ',
    '😉': ' vui ', '🫣': ' vui ',

    # --- Tiêu cực / Thất vọng ---
    '😭': ' xấu ', '😢': ' buồn ', '😥': ' buồn ', '😿': ' buồn ',
    '🥺': ' buồn ', '😞': ' buồn ', '😔': ' buồn ', '😩': ' buồn ',
    '😫': ' buồn ', '😪': ' buồn ', '😰': ' buồn ',
    '😓': ' buồn ', '😟': ' buồn ', '🙁': ' buồn ', '☹️': ' buồn ',


    '😡': ' xấu ', '🤬': ' xấu ', '😠': ' xấu ',
    '👎': ' xấu ', '👎🏻': ' xấu ', '👎🏼': ' xấu ',
    '💔': ' xấu ', '🖕': ' xấu ',


    '🤮': ' xấu ', '👿': ' xấu ', '😤': ' xấu ',
    '💩': ' xấu ', '🗑️': ' xấu ',


    '😱': ' ngạc_nhiên ', '😲': ' ngạc_nhiên ',
    '🤔': ' suy_nghĩ ', '🧐': ' suy_nghĩ ',
    '😐': ' bình_thường ', '😑': ' bình_thường ', '😶': ' bình_thường ',


    '⭐': ' năm_sao ', '🌟': ' năm_sao ', '💫': ' năm_sao ',


    '🎶': ' vui ', '🎵': ' vui ', '🎼': ' vui ',

    # --- Biểu tượng dạng text (emoticon) ---
    '<3': ' tốt ', ':3': ' vui ', ':)': ' vui ', ':))': ' vui ',
    ':)))': ' vui ', ':(': ' buồn ', ':((': ' buồn ',
    ':(((': ' buồn ', ':D': ' vui ', 'XD': ' vui ',
    '^_^': ' vui ', '^^': ' vui ', '>_<': ' buồn ',
    '*_*': ' tốt ', 'T_T': ' buồn ', '-_-': ' bình_thường ',
    ':P': ' vui ', ':p': ' vui ', ';)': ' vui ',
    'o_O': ' ngạc_nhiên ', 'O_o': ' ngạc_nhiên ',
}


# ============================================================================
# 2. TỪ ĐIỂN TEENCODE & VIẾT TẮT THƯƠNG MẠI ĐIỆN TỬ
# ============================================================================
TEENCODE_MAP = {

    'k': 'không', 'ko': 'không', 'khg': 'không', 'kh': 'không',
    'khong': 'không', 'kg': 'không', 'hk': 'không', 'hkg': 'không',
    'kp': 'không phải', 'kfai': 'không phải', 'kphai': 'không phải',
    'hem': 'không', 'hông': 'không', 'hong': 'không',
    'éo': 'không', 'đéo': 'không',
    'chx': 'chưa',
    'cx': 'cũng', 'cg': 'cũng', 'cug': 'cũng', 'cũg': 'cũng',
    'chg': 'chẳng',


    'mng': 'mọi người', 'mn': 'mọi người', 'm.n': 'mọi người',
    'mk': 'mình', 'mik': 'mình', 'mh': 'mình', 'mih': 'mình',
    'minh': 'mình', 'mìk': 'mình', 'minhk': 'mình',
    'nv': 'nhân viên',
    'bn': 'bạn',
    'tui': 'tôi',
    'ny': 'người yêu', 'ck': 'chồng', 'vk': 'vợ',
    'ng': 'người',


    'j': 'gì', 'gi': 'gì', 'gj': 'gì',
    'z': 'vậy', 'dz': 'vậy', 'v': 'vậy', 'vay': 'vậy',


    'dc': 'được', 'dk': 'được', 'đc': 'được', 'đk': 'được',
    'duoc': 'được', 'đuợc': 'được', 'đươc': 'được',
    'r': 'rồi', 'rui': 'rồi', 'ròi': 'rồi', 'roi': 'rồi',
    'lun': 'luôn', 'luon': 'luôn',
    'lm': 'làm', 'lam': 'làm',
    'bt': 'bình thường', 'bthg': 'bình thường', 'bth': 'bình thường',
    'bthường': 'bình thường',
    'vs': 'với', 'voi': 'với',
    'ntn': 'như thế này', 'nvay': 'như vậy',
    'w': 'quá', 'wa': 'quá', 'qá': 'quá', 'wá': 'quá',
    'nhiu': 'nhiều', 'nhìu': 'nhiều', 'nhều': 'nhiều',
    'nhahh': 'nhanh',


    'tks': 'cảm ơn', 'tk': 'cảm ơn', 'thanks': 'cảm ơn',
    'thank': 'cảm ơn', 'thankss': 'cảm ơn', 'thenks': 'cảm ơn',
    'thks': 'cảm ơn',
    'sr': 'xin lỗi', 'sorry': 'xin lỗi', 'sory': 'xin lỗi',


    'ok': 'tốt', 'oke': 'tốt', 'okie': 'tốt', 'okz': 'tốt',
    'okela': 'tốt', 'oki': 'tốt', 'okay': 'tốt',
    'gud': 'tốt', 'good': 'tốt', 'gooddd': 'tốt',
    'gd': 'tốt', 'goood': 'tốt',
    'nice': 'tốt', 'niceee': 'tốt',
    'perfect': 'hoàn hảo', 'great': 'tuyệt vời',
    'super': 'siêu', 'excellent': 'xuất sắc',
    'recommend': 'khuyên dùng', 'bad': 'xấu',
    'xau': 'xấu', 'tot': 'tốt', 'dep': 'đẹp',
    'đep': 'đẹp', 'depp': 'đẹp', 'dẹp': 'đẹp',
    'thik': 'thích', 'thix': 'thích',
    'ưg': 'ưng', 'ug': 'ưng',
    'siu': 'siêu', 'ciu': 'buồn',
    'hịn': 'đẹp', 'ghiền': 'thích',


    'hi': 'vui', 'hii': 'vui', 'hiii': 'vui',
    'hihi': 'vui', 'hehe': 'vui',
    'huhu': 'buồn', 'huhuu': 'buồn', 'huhuhu': 'buồn',
    'hic': 'buồn', 'hicc': 'buồn',


    'shop': 'cửa hàng', 'sp': 'sản phẩm',
    'spm': 'sản phẩm', 'sanpham': 'sản phẩm',
    'ship': 'giao hàng', 'shipper': 'người giao hàng',
    'shiper': 'người giao hàng', 'shoppe': 'shopee',
    'dt': 'điện thoại',
    'sale': 'giảm giá',
    'sz': 'size', 'soze': 'size',
    'ncc': 'nhà cung cấp',
    'hsd': 'hạn sử dụng', 'nsx': 'ngày sản xuất',
    'vc': 'vận chuyển', 'ghtk': 'giao hàng tiết kiệm',


    'hàg': 'hàng', 'hag': 'hàng', 'hang': 'hàng',


    'rep': 'trả lời', 'ib': 'nhắn tin',
    'inbox': 'nhắn tin', 'nt': 'nhắn tin',
    'onl': 'trực tuyến', 'online': 'trực tuyến',
    'fb': 'phản hồi', 'nch': 'nói chuyện',


    'ncl': 'nói chung là', 'nchung': 'nói chung',
    'trc': 'trước', 'truoc': 'trước',
    'nua': 'nữa', 'nưa': 'nữa',
    'lg': 'lượng', 'cl': 'chất lượng',
    'potay': 'bó tay',


    'value': 'giá trị', 'money': 'tiền',
    'quality': 'chất lượng', 'price': 'giá',
    'delivery': 'giao hàng', 'fast': 'nhanh',
    'slow': 'chậm', 'beautiful': 'đẹp',
    'ugly': 'xấu', 'cheap': 'rẻ', 'expensive': 'đắt',
    'love': 'yêu', 'hate': 'ghét', 'like': 'thích',
    'fake': 'giả', 'real': 'thật',


    'form': 'dáng', 'foem': 'dáng', 'from': 'dáng', 'phom': 'dáng',
    'fai': 'phải', 'fải': 'phải',
    'dã man': 'cực kỳ',
    'vãi': 'rất', 'vại': 'rất',
    'khỏi chê': 'rất tốt', 'hết sảy': 'rất tốt',
    'chán đời': 'rất xấu',


    '5sao': 'năm sao', '5*': 'năm sao', '4*': 'bốn sao',
    '3*': 'ba sao', '2*': 'hai sao', '1*': 'một sao',
    '10đ': 'mười điểm', '5s': 'năm sao',
}


# ============================================================================
# 3. TỪ ĐIỂN SỬA LỖI CHÍNH TẢ PHỔ BIẾN
# ============================================================================
SPELLING_CORRECTIONS = {

    'sản phr': 'sản phẩm', 'sản fẩm': 'sản phẩm', 'sảm phẩm': 'sản phẩm',
    'sản phầm': 'sản phẩm', 'sản phẩn': 'sản phẩm', 'sản phản': 'sản phẩm',
    'san phạm': 'sản phẩm', 'sản phẩ': 'sản phẩm',


    'chất lượg': 'chất lượng', 'chất luợng': 'chất lượng',
    'chất luong': 'chất lượng', 'chất lương': 'chất lượng',
    'chat luong': 'chất lượng',


    'giao hàg': 'giao hàng', 'giao hang': 'giao hàng',
    'giao hành': 'giao hàng',


    'tuyệt vờ': 'tuyệt vời', 'tuyệt vờii': 'tuyệt vời',
    'tuyet voi': 'tuyệt vời', 'tuyệt vờiO': 'tuyệt vời',


    'đóng goi': 'đóng gói', 'dong goi': 'đóng gói', 'đóng gỏi': 'đóng gói',


    'chăc chăn': 'chắc chắn', 'chắc chắc': 'chắc chắn', 'chac chan': 'chắc chắn',


    'nhiet tinh': 'nhiệt tình', 'nhiệt tinhg': 'nhiệt tình',


    'hài lòg': 'hài lòng', 'hai long': 'hài lòng',


    'phục vu': 'phục vụ', 'phụ vụ': 'phục vụ', 'phuc vu': 'phục vụ',


    'thất vọg': 'thất vọng', 'that vong': 'thất vọng',


    'uy tin': 'uy tín',


    'giống hjnh': 'giống hình', 'giống hinh': 'giống hình',
    'giong hinh': 'giống hình',


    'sử dụg': 'sử dụng', 'su dung': 'sử dụng', 'sủ dụng': 'sử dụng',


    'đág tiền': 'đáng tiền', 'dang tien': 'đáng tiền',


    'cam on': 'cảm ơn', 'cám ơn': 'cảm ơn',


    'day dan': 'dày dặn', 'dày dăn': 'dày dặn',


    'vận chuyên': 'vận chuyển', 'van chuyen': 'vận chuyển',


    'co gian': 'co giãn', 'co giản': 'co giãn',


    'kich thuoc': 'kích thước', 'kich thươc': 'kích thước',


    'mem min': 'mềm mịn',


    'tương sứng': 'tương xứng',


    'dỗm': 'kém chất lượng', 'dom': 'kém chất lượng',
    'hãm': 'tệ', 'hớ': 'bị lừa',
    'sácg': 'sách', 'mảu': 'màu',
    'chât vải': 'chất vải', 'chat vai': 'chất vải',
    'thời gjàn': 'thời gian', 'thoi gian': 'thời gian',
    'chup': 'chụp',
}


# ============================================================================
# 4. TỪ ĐIỂN VIẾT DÍNH CHỮ PHỔ BIẾN
# ============================================================================
CONCAT_PATTERNS = [

    (r'giaohàng', 'giao hàng'), (r'giaohang', 'giao hàng'),
    (r'sảnphẩm', 'sản phẩm'), (r'sanpham', 'sản phẩm'),
    (r'chấtlượng', 'chất lượng'), (r'chatluong', 'chất lượng'),
    (r'đónggói', 'đóng gói'), (r'donggoi', 'đóng gói'),
    (r'tuyệtvời', 'tuyệt vời'), (r'tuyetvoi', 'tuyệt vời'),
    (r'hàilòng', 'hài lòng'), (r'hailong', 'hài lòng'),
    (r'cửahàng', 'cửa hàng'), (r'cuahang', 'cửa hàng'),
    (r'giátiền', 'giá tiền'), (r'giatien', 'giá tiền'),
    (r'phùhợp', 'phù hợp'), (r'phuhop', 'phù hợp'),


    (r'rấtđẹp', 'rất đẹp'), (r'ratdep', 'rất đẹp'),
    (r'rấttốt', 'rất tốt'), (r'rattot', 'rất tốt'),
    (r'khôngtốt', 'không tốt'), (r'khongtot', 'không tốt'),
    (r'đẹplắm', 'đẹp lắm'), (r'deplam', 'đẹp lắm'),
    (r'xấulắm', 'xấu lắm'),


    (r'thờigian', 'thời gian'), (r'thoigian', 'thời gian'),
    (r'đángtiền', 'đáng tiền'), (r'dangtien', 'đáng tiền'),
    (r'ủnghộ', 'ủng hộ'), (r'ungho', 'ủng hộ'),
    (r'nhiệttình', 'nhiệt tình'), (r'nhiệttinh', 'nhiệt tình'),
    (r'cẩnthận', 'cẩn thận'), (r'canthan', 'cẩn thận'),
    (r'chắcchắn', 'chắc chắn'), (r'chacchan', 'chắc chắn'),
    (r'thoảimái', 'thoải mái'), (r'thoaimai', 'thoải mái'),


    (r'thấtvọng', 'thất vọng'), (r'thatvong', 'thất vọng'),
    (r'vuivẻ', 'vui vẻ'),
    (r'buồncười', 'buồn cười'),
    (r'mấttiền', 'mất tiền'),
    (r'khỏichê', 'không chê được'),
]


# ============================================================================
# 5. STOPWORDS TIẾNG VIỆT (từ chức năng, không mang cảm xúc)
# ============================================================================
VIETNAMESE_STOPWORDS = {
    'và', 'của', 'có', 'là', 'cho', 'được', 'đã', 'hay',
    'các', 'này', 'với', 'thì', 'đó', 'để', 'khi',
    'từ', 'theo', 'một', 'những', 'do', 'bởi',
    'nên', 'vì', 'tại', 'bị', 'về', 'trong', 'trên',
    'dưới', 'ra', 'vào', 'lên', 'xuống', 'đi', 'đến',
    'nơi', 'đây', 'kia', 'ấy', 'nọ',
    'cùng', 'cứ', 'thế', 'vẫn', 'ngay',
    'ạ', 'à', 'ơi', 'nhé', 'nha', 'nhá', 'nè', 'thôi',
}


# ============================================================================
#                    CÁC HÀM XỬ LÝ
# ============================================================================

def replace_emojis(text):

    text_emoticons_sorted = sorted(
        [(k, v) for k, v in EMOJI_MAP.items()
         if all(c in ':<>()_^*3DPpXxTtOo;/-' or c.isalpha() for c in k)],
        key=lambda x: len(x[0]), reverse=True
    )
    for emoticon, replacement in text_emoticons_sorted:
        text = text.replace(emoticon, replacement)

    for emoji_char, replacement in EMOJI_MAP.items():
        if emoji_char in text:
            text = text.replace(emoji_char, replacement)

    return text


def normalize_unicode(text):

    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    return text


def remove_noise(text):

    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    text = re.sub(r'<[^>]+>', ' ', text)

    text = re.sub(r'\S+@\S+\.\S+', ' ', text)

    text = re.sub(r'[#@]\w+', ' ', text)

    text = re.sub(r'\d+[₫đ]', ' ', text)
    text = re.sub(r'\d+[\.,]\d+', ' ', text)

    text = re.sub(
        r'[^\sa-zA-Z0-9_'
        r'àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩ'
        r'òóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ'
        r'ÀÁẠẢÃÂẦẤẬẨẪĂẰẮẶẲẴÈÉẸẺẼÊỀẾỆỂỄÌÍỊỈĨ'
        r'ÒÓỌỎÕÔỒỐỘỔỖƠỜỚỢỞỠÙÚỤỦŨƯỪỨỰỬỮỲÝỴỶỸĐ]',
        ' ', text
    )

    text = re.sub(r'\b\d+\b', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()
    return text

def reduce_elongation(text):

    text = re.sub(r'([a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ])\1{2,}', r'\1', text)

    text = re.sub(r'([aeiouyàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹ])\1{1,}', r'\1', text)
    return text

def fix_common_spelling(text):

    sorted_corrections = sorted(SPELLING_CORRECTIONS.items(), key=lambda x: len(x[0]), reverse=True)
    for wrong, correct in sorted_corrections:

        pattern = r'\b' + re.escape(wrong) + r'\b'
        text = re.sub(pattern, correct, text)
    return text


def fix_concatenated_words(text):

    for pattern, replacement in CONCAT_PATTERNS:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


def map_teencode(text):

    words = text.split()
    result = []
    i = 0
    while i < len(words):
        matched = False
        # Thử ghép 3 từ → 2 từ → 1 từ (ưu tiên cụm dài hơn)
        for n in range(min(3, len(words) - i), 0, -1):
            phrase = ' '.join(words[i:i + n])
            if phrase in TEENCODE_MAP:
                result.append(TEENCODE_MAP[phrase])
                i += n
                matched = True
                break
        if not matched:
            result.append(words[i])
            i += 1
    return ' '.join(result)


def tokenize_vietnamese(text):

    try:
        from underthesea import word_tokenize
        text = word_tokenize(text, format="text")
    except ImportError:
        print("⚠️ Thư viện underthesea chưa được cài. Bỏ qua bước tách từ.")
    except Exception:
        # Nếu underthesea gặp lỗi với câu quá ngắn hoặc ký tự lạ → giữ nguyên
        pass
    return text


def remove_stopwords(text, stopwords_set):
    words = text.split()
    filtered = [w for w in words if w.replace('_', ' ') not in stopwords_set and w not in stopwords_set]
    return ' '.join(filtered)


def remove_short_tokens(text, min_length=2):
    meaningful_single = {'đẹp', 'xấu', 'tốt', 'rẻ', 'dở', 'ổn', 'êm', 'ẩu'}
    words = text.split()
    filtered = [w for w in words if len(w.replace('_', '')) >= min_length or w in meaningful_single]
    return ' '.join(filtered)


# ============================================================================
#              PIPELINE TỔNG HỢP - GỌI TẤT CẢ CÁC GIAI ĐOẠN
# ============================================================================

def full_preprocess(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""

    # 1. Chuẩn hóa Unicode + lowercase
    text = normalize_unicode(text)

    # 2. Thay thế Emoji → văn bản (TRƯỚC khi xóa ký tự đặc biệt)
    text = replace_emojis(text)

    # 3. Loại bỏ URL, HTML, ký tự đặc biệt
    text = remove_noise(text)

    # 4. Rút gọn ký tự kéo dài ("đẹpppp" → "đẹp")
    text = reduce_elongation(text)

    # 5. Sửa lỗi chính tả phổ biến
    text = fix_common_spelling(text)

    # 6a. Sửa lỗi viết dính chữ ("giaohàng" → "giao hàng")
    text = fix_concatenated_words(text)

    # 6b. Ánh xạ teencode & từ viết tắt ("ko dc" → "không được")
    text = map_teencode(text)

    # 7. Tách từ tiếng Việt ("sản phẩm" → "sản_phẩm")
    text = tokenize_vietnamese(text)

    # 8. (Tùy chọn) Loại bỏ stopwords
    if USE_STOPWORDS:
        text = remove_stopwords(text, VIETNAMESE_STOPWORDS)

    # 9. Loại bỏ token quá ngắn (1 ký tự vô nghĩa)
    text = remove_short_tokens(text)

    # 10. Chuẩn hóa khoảng trắng cuối cùng
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# ============================================================================
#                           THỰC THI CHÍNH
# ============================================================================

print("=" * 70)
print("  PIPELINE TIỀN XỬ LÝ DỮ LIỆU PHÂN TÍCH CẢM XÚC TIẾNG VIỆT")
print("  TF-IDF + SVM | Kaggle Vietnamese Sentiment Analysis")
print("=" * 70)

# ----------------------------------------------------------------------
# BƯỚC 0: Cài đặt thư viện Underthesea (tự động trên Kaggle)
# ----------------------------------------------------------------------
print("\n📦 [0/7] Kiểm tra & cài đặt thư viện underthesea...")
try:
    from underthesea import word_tokenize
    print("   ✅ underthesea đã sẵn sàng.")
except ImportError:
    print("   ⏳ Đang cài đặt underthesea (mất ~1-2 phút)...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'underthesea', '-q'])
    from underthesea import word_tokenize
    print("   ✅ Cài đặt underthesea thành công.")

# ----------------------------------------------------------------------
# BƯỚC 1: Đọc dữ liệu
# ----------------------------------------------------------------------
print(f"\n📂 [1/7] Đang đọc file: {DATA_PATH}")
df = pd.read_csv(DATA_PATH, encoding='utf-8')
print(f"   ✅ Đọc thành công: {len(df):,} dòng × {len(df.columns)} cột")
print(f"   📋 Các cột: {list(df.columns)}")

# ----------------------------------------------------------------------
# BƯỚC 2: Kiểm tra & làm sạch dữ liệu cơ bản
# ----------------------------------------------------------------------
print(f"\n🔍 [2/7] Kiểm tra chất lượng dữ liệu...")

# Đổi tên cột cho thống nhất (file gốc có cột: content, label, start)
if 'content' in df.columns:
    df.rename(columns={'content': 'comment'}, inplace=True)
if 'start' in df.columns:
    df.rename(columns={'start': 'rate'}, inplace=True)

# Kiểm tra null
null_counts = df.isnull().sum()
print(f"   📊 Giá trị null:\n{null_counts.to_string()}")

# Xóa dòng có comment null hoặc rỗng
original_len = len(df)
df = df.dropna(subset=['comment'])
df = df[df['comment'].str.strip().str.len() > 0]
dropped = original_len - len(df)
if dropped > 0:
    print(f"   ⚠️ Đã xóa {dropped} dòng rỗng/null")

# Xóa dòng trùng lặp
dup_count = df.duplicated(subset=['comment']).sum()
df = df.drop_duplicates(subset=['comment'], keep='first')
if dup_count > 0:
    print(f"   ⚠️ Đã xóa {dup_count} dòng trùng lặp")

print(f"   ✅ Còn lại: {len(df):,} dòng")

# ----------------------------------------------------------------------
# BƯỚC 3: Chuẩn hóa nhãn
# ----------------------------------------------------------------------
print(f"\n🏷️ [3/7] Chuẩn hóa nhãn cảm xúc...")
df['label'] = df['label'].str.strip().str.upper()

# Hiển thị phân phối nhãn dạng biểu đồ text
label_dist = df['label'].value_counts()
print(f"   📊 Phân phối nhãn:")
for label, count in label_dist.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"      {label:>4s}: {count:>6,} ({pct:5.1f}%) {bar}")


# ----------------------------------------------------------------------
# BƯỚC 4: Hiển thị mẫu dữ liệu GỐC
# ----------------------------------------------------------------------
print(f"\n📝 [4/7] Mẫu dữ liệu GỐC (10 dòng đầu):")
print("-" * 70)
for i, row in df.head(10).iterrows():
    comment = str(row['comment'])[:120]
    label = row['label']
    print(f"  [{label:>3s}] {comment}")
print("-" * 70)

# ----------------------------------------------------------------------
# BƯỚC 5: ÁP DỤNG PIPELINE TIỀN XỬ LÝ
# ----------------------------------------------------------------------
print(f"\n⚙️ [5/7] Đang chạy pipeline tiền xử lý trên {len(df):,} bình luận...")
print("   (Bước này có thể mất 5-15 phút tùy kích thước dữ liệu)")

start_time = time.time()

# === ÁP DỤNG HÀM PIPELINE CHO TỪNG DÒNG ===
df['cleaned_comment'] = df['comment'].apply(full_preprocess)

elapsed = time.time() - start_time
print(f"   ✅ Hoàn thành trong {elapsed:.1f} giây ({elapsed / 60:.1f} phút)")
print(f"   ⏱️ Tốc độ: {len(df) / elapsed:.0f} bình luận/giây")

# ----------------------------------------------------------------------
# BƯỚC 6: KIỂM TRA KẾT QUẢ & THỐNG KÊ
# ----------------------------------------------------------------------
print(f"\n📊 [6/7] Thống kê sau tiền xử lý:")
print("-" * 70)

# Thống kê độ dài ký tự
df['orig_len'] = df['comment'].str.len()
df['clean_len'] = df['cleaned_comment'].str.len()
print(f"  Độ dài trung bình (ký tự):")
print(f"    Trước xử lý: {df['orig_len'].mean():.0f} ± {df['orig_len'].std():.0f}")
print(f"    Sau xử lý:   {df['clean_len'].mean():.0f} ± {df['clean_len'].std():.0f}")
compression = (1 - df['clean_len'].sum() / df['orig_len'].sum()) * 100
print(f"    Tỷ lệ nén:   {compression:.1f}%")

# Thống kê số token
df['orig_tokens'] = df['comment'].str.split().str.len()
df['clean_tokens'] = df['cleaned_comment'].str.split().str.len()
print(f"\n  Số token trung bình:")
print(f"    Trước xử lý: {df['orig_tokens'].mean():.1f} token/bình luận")
print(f"    Sau xử lý:   {df['clean_tokens'].mean():.1f} token/bình luận")

# Kiểm tra dòng rỗng sau xử lý
empty_after = (df['cleaned_comment'].str.strip().str.len() == 0).sum()
print(f"\n  Dòng rỗng sau xử lý: {empty_after} ({empty_after / len(df) * 100:.2f}%)")

# --- Hiển thị mẫu so sánh TRƯỚC và SAU ---
print(f"\n📝 Mẫu so sánh TRƯỚC và SAU tiền xử lý:")
print("=" * 70)

# Chọn các mẫu có teencode, emoji, lỗi chính tả để demo rõ hiệu quả
interesting_indices = []
for idx, row in df.iterrows():
    comment = str(row['comment']).lower()
    if any(tc in comment for tc in ['ko ', ' k ', 'sp ', 'dc ', 'đc ', 'thik', 'oke',
                                     'ship', '❤', '😍', '👍', ':))', '<3', 'hiii',
                                     'tks', 'sản phr', 'chất lượg']):
        interesting_indices.append(idx)
    if len(interesting_indices) >= 20:
        break

# Bổ sung mẫu ngẫu nhiên nếu chưa đủ 20
if len(interesting_indices) < 20:
    remaining = df.index.difference(interesting_indices).tolist()
    import random
    random.seed(42)
    interesting_indices.extend(random.sample(remaining, min(20 - len(interesting_indices), len(remaining))))

for idx in interesting_indices[:20]:
    row = df.loc[idx]
    print(f"\n  [{row['label']:>3s}] GỐC:  {str(row['comment'])[:130]}")
    print(f"        SẠCH: {str(row['cleaned_comment'])[:130]}")
    print(f"  {'─' * 66}")

# ----------------------------------------------------------------------
# BƯỚC 7: LƯU FILE KẾT QUẢ
# ----------------------------------------------------------------------
print(f"\n💾 [7/7] Đang lưu file kết quả...")

# File đầy đủ (comment gốc + label + comment đã sạch + rate)
output_df = df[['comment', 'label', 'cleaned_comment']].copy()
if 'rate' in df.columns:
    output_df['rate'] = df['rate']

output_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f"   ✅ Đã lưu: {OUTPUT_PATH}")
print(f"   📊 Kích thước: {len(output_df):,} dòng × {len(output_df.columns)} cột")

# File 2 cột sẵn sàng cho huấn luyện TF-IDF + SVM
train_output = 'cleaned_data_train_ready.csv'
output_df[['cleaned_comment', 'label']].to_csv(train_output, index=False, encoding='utf-8-sig')
print(f"   ✅ Đã lưu bản huấn luyện: {train_output}")

# ----------------------------------------------------------------------
# TỔNG KẾT
# ----------------------------------------------------------------------
print("\n" + "=" * 70)
print("  ✅ PIPELINE TIỀN XỬ LÝ HOÀN TẤT!")
print("=" * 70)
print(f"""
  📋 Tóm tắt:
  ├── Tổng bình luận:        {len(output_df):>8,}
  ├── Phân phối nhãn:""")
for label, count in label_dist.items():
    print(f"  │   ├── {label}:             {count:>8,} ({count / len(df) * 100:.1f}%)")
print(f"""  ├── Độ dài TB (token):     {df['clean_tokens'].mean():>8.1f}
  ├── File output:           {OUTPUT_PATH}
  └── File train-ready:      {train_output}

  📌 Các bước tiền xử lý đã áp dụng:
     1. ✅ Chuẩn hóa Unicode + lowercase
     2. ✅ Số hóa Emoji → văn bản cảm xúc
     3. ✅ Loại bỏ URL, HTML, ký tự đặc biệt
     4. ✅ Rút gọn ký tự kéo dài (đẹpppp → đẹp)
     5. ✅ Sửa lỗi chính tả phổ biến
     6. ✅ Sửa lỗi viết dính chữ
     7. ✅ Ánh xạ Teencode & từ viết tắt TMĐT (~150 mapping)
     8. ✅ Tách từ tiếng Việt (Underthesea)
     9. {'✅' if USE_STOPWORDS else '⏩'} Loại bỏ stopwords {'(BẬT)' if USE_STOPWORDS else '(TẮT)'}
    10. ✅ Loại bỏ token ngắn vô nghĩa
""")

  PIPELINE TIỀN XỬ LÝ DỮ LIỆU PHÂN TÍCH CẢM XÚC TIẾNG VIỆT
  TF-IDF + SVM | Kaggle Vietnamese Sentiment Analysis

📦 [0/7] Kiểm tra & cài đặt thư viện underthesea...
   ✅ underthesea đã sẵn sàng.

📂 [1/7] Đang đọc file: /content/data.csv
   ✅ Đọc thành công: 31,460 dòng × 3 cột
   📋 Các cột: ['content', 'label', 'start']

🔍 [2/7] Kiểm tra chất lượng dữ liệu...
   📊 Giá trị null:
comment    24
label       0
rate        0
   ⚠️ Đã xóa 24 dòng rỗng/null
   ⚠️ Đã xóa 4920 dòng trùng lặp
   ✅ Còn lại: 26,516 dòng

🏷️ [3/7] Chuẩn hóa nhãn cảm xúc...
   📊 Phân phối nhãn:
       POS: 15,990 ( 60.3%) ██████████████████████████████
       NEG:  6,300 ( 23.8%) ███████████
       NEU:  4,226 ( 15.9%) ███████

📝 [4/7] Mẫu dữ liệu GỐC (10 dòng đầu):
----------------------------------------------------------------------
  [POS] Áo bao đẹp ạ!
  [POS] Tuyệt vời
  [NEG] 2day ao khong giong trong
  [POS] Mùi thơm,bôi lên da mềm da
  [POS] Vải đẹp, dày dặn
  [POS] Hàng rất đẹp, rất chi là ưng ý
  [POS] Chấ

In [ ]:
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np


class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, preprocess_func):
        self.preprocess_func = preprocess_func

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        if isinstance(X, pd.Series):
            X_processed = X.apply(self.preprocess_func)
        elif isinstance(X, np.ndarray) and X.ndim == 1:
            X_processed = np.array([self.preprocess_func(text) for text in X])
        elif isinstance(X, list):
            X_processed = [self.preprocess_func(text) for text in X]
        else:
            raise TypeError("Đầu vào X phải là một chuỗi (pandas Series), một mảng numpy, hoặc một danh sách các chuỗi.")
        return X_processed


print("⏳ Giai đoạn 1: Đang chia tập dữ liệu Train/Test...")

X = df['comment']
y = df['label']


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"✅ Chia dữ liệu thành công!")
print(f"   - Kích thước tập Train (X_train): {X_train.shape}")
print(f"   - Kích thước tập Test  (X_test) : {X_test.shape}\n")


print("⏳ Giai đoạn 2: Khởi tạo Pipeline (TextPreprocessor + TF-IDF + SVM) và cấu hình tham số GridSearch...")


pipeline = Pipeline([
    ('preprocessor', TextPreprocessor(preprocess_func=full_preprocess)),
    ('tfidf', TfidfVectorizer()),
    ('svm', LinearSVC(random_state=42, class_weight='balanced', dual='auto'))
])

param_grid = [
    {
        'tfidf__analyzer': ['word'],
        'tfidf__max_features': [20000],
        'tfidf__ngram_range': [(1, 2)],
        'svm__C': [0.1, 1, 10]
    },
    {
        'tfidf__analyzer': ['char_wb'],
        'tfidf__ngram_range': [(2, 4)],
        'tfidf__max_features': [20000],
        'svm__C': [0.1, 1, 10]
    }
]


grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)
print("✅ Cấu hình GridSearch nâng cao sẵn sàng.\n")


print("⏳ Giai đoạn 3: Đang tiến hành huấn luyện GridSearch (Quá trình phân tích N-gram và Char-level có thể mất chút thời gian)...")


grid_search.fit(X_train, y_train)

print("✅ Huấn luyện hoàn tất!")
print(f"   - Tham số tốt nhất (Best Params): {grid_search.best_params_}")
print(f"   - Điểm F1-weighted tốt nhất trên tập Train: {grid_search.best_score_:.4f}\n")


print("⏳ Giai đoạn 4: Đang dự đoán và đánh giá hiệu năng trên tập Test...")


best_pipeline = grid_search.best_estimator_


y_pred_final = best_pipeline.predict(X_test)


final_accuracy = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final, average='weighted')
final_recall = recall_score(y_test, y_pred_final, average='weighted')
final_f1 = f1_score(y_test, y_pred_final, average='weighted')
final_report = classification_report(y_test, y_pred_final)

output_str = """
======================================================================
🏆 KẾT QUẢ : PIPELINE (TF-IDF + SVM N-GRAM OPTIMIZED)
======================================================================
📊 KẾT QUẢ TRÊN TẬP KIỂM TRA (TEST SET):
Accuracy : {:.4f}
Precision: {:.4f}
Recall   : {:.4f}
F1-score : {:.4f}

📋 Chi tiết Classification Report:
{}
""".format(final_accuracy, final_precision, final_recall, final_f1, final_report)

print(output_str)


print("⏳ Giai đoạn 5: Đang lưu mô hình xuống đĩa cứng...")


joblib.dump(best_pipeline, "tfidf_svm_pipeline.pkl")

print("🎯 THÀNH CÔNG! File 'tfidf_svm_pipeline.pkl' đã được lưu với cấu hình nâng cao.")

⏳ Giai đoạn 1: Đang chia tập dữ liệu Train/Test...
✅ Chia dữ liệu thành công!
   - Kích thước tập Train (X_train): (21212,)
   - Kích thước tập Test  (X_test) : (5304,)

⏳ Giai đoạn 2: Khởi tạo Pipeline (TextPreprocessor + TF-IDF + SVM) và cấu hình tham số GridSearch...
✅ Cấu hình GridSearch nâng cao sẵn sàng.

⏳ Giai đoạn 3: Đang tiến hành huấn luyện GridSearch (Quá trình phân tích N-gram và Char-level có thể mất chút thời gian)...
Fitting 3 folds for each of 6 candidates, totalling 18 fits


In [ ]:
import joblib

#============================================================================
# DỰ ĐOÁN THỬ NGHIỆM
# ============================================================================

print("\n" + "="*70)
print("🧪 DEMO DỰ ĐOÁN")
print("="*70)


my_model = joblib.load("tfidf_svm_pipeline.pkl")

samples = [
    "sản phẩm rất tốt giao hàng nhanh đóng gói đẹp",
    "quá tệ thất vọng chất lượng kém",
    "shop phục vụ nhiệt tình sẽ ủng hộ tiếp",
        "Thấy bình thường",
    "hàng xấu không giống hình",
    "🫶"
]

for text in samples:

    processed_text = full_preprocess(text)

    pred = my_model.predict([processed_text])[0]

    print(f"\n📝 Bình luận: {text}")
    print(f"👉 Dự đoán cảm xúc: {pred}")

In [ ]:
from google.colab import files
files.download("tfidf_svm_pipeline.pkl")